# 1. 데이터셋 준비
> 캐글에 올려둔 train, val 데이터셋 불러옴
> 
> JSON 라벨 파일을 YOLOv8이 읽을 수 있는 형식으로 변환
>
> 학습 전 환경 세팅

In [3]:
# =====================
# 0. 라이브러리 설치
# =====================
import subprocess
subprocess.run(['pip', 'install', 'ultralytics', '-q'])

import os
import json

# =====================
# 1. 디렉토리 생성
# =====================
os.makedirs('/kaggle/working/dataset/labels/train', exist_ok=True)
os.makedirs('/kaggle/working/dataset/labels/val', exist_ok=True)
os.makedirs('/kaggle/working/boar/images/train', exist_ok=True)
os.makedirs('/kaggle/working/boar/images/val', exist_ok=True)

# =====================
# 2. bbox 변환 함수
# =====================
def convert_bbox_to_yolo(bbox, img_w, img_h):
    x1, y1 = bbox[0]
    x2, y2 = bbox[1]
    center_x = (x1 + x2) / 2 / img_w
    center_y = (y1 + y2) / 2 / img_h
    width = (x2 - x1) / img_w
    height = (y2 - y1) / img_h
    return center_x, center_y, width, height

# =====================
# 3. train 라벨 변환
# =====================
tl_path = '/kaggle/input/datasets/hyejeong12/boar-train/TL_02.'
ts_path = '/kaggle/input/datasets/hyejeong12/boar-train/TS_02.'
label_output = '/kaggle/working/dataset/labels/train'

json_files = os.listdir(tl_path)
success, skip = 0, 0

for json_file in json_files:
    json_path = os.path.join(tl_path, json_file)
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    img_info = data['images'][0]
    img_w = img_info['width']
    img_h = img_info['height']
    file_name = img_info['file_name']
    
    if not os.path.exists(os.path.join(ts_path, file_name)):
        skip += 1
        continue
    
    label_lines = []
    for ann in data['annotations']:
        if ann['drawing_id'] != 1:
            continue
        cx, cy, w, h = convert_bbox_to_yolo(ann['bbox'], img_w, img_h)
        if not all(0 <= v <= 1 for v in [cx, cy, w, h]):
            skip += 1
            continue
        label_lines.append(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    
    if not label_lines:
        skip += 1
        continue
    
    txt_name = os.path.splitext(file_name)[0] + '.txt'
    with open(os.path.join(label_output, txt_name), 'w') as f:
        f.write('\n'.join(label_lines))
    success += 1

print(f"train 변환 완료 - 성공: {success}, 건너뜀: {skip}")

# =====================
# 4. val 라벨 변환
# =====================
vl_path = '/kaggle/input/datasets/hyunseo2/boar-val/VL_02.'
vs_path = '/kaggle/input/datasets/hyunseo2/boar-val/VS_02.'
label_output_val = '/kaggle/working/dataset/labels/val'

json_files = os.listdir(vl_path)
success, skip = 0, 0

for json_file in json_files:
    json_path = os.path.join(vl_path, json_file)
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    img_info = data['images'][0]
    img_w = img_info['width']
    img_h = img_info['height']
    file_name = img_info['file_name']
    
    if not os.path.exists(os.path.join(vs_path, file_name)):
        skip += 1
        continue
    
    label_lines = []
    for ann in data['annotations']:
        if ann['drawing_id'] != 1:
            continue
        cx, cy, w, h = convert_bbox_to_yolo(ann['bbox'], img_w, img_h)
        if not all(0 <= v <= 1 for v in [cx, cy, w, h]):
            skip += 1
            continue
        label_lines.append(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    
    if not label_lines:
        skip += 1
        continue
    
    txt_name = os.path.splitext(file_name)[0] + '.txt'
    with open(os.path.join(label_output_val, txt_name), 'w') as f:
        f.write('\n'.join(label_lines))
    success += 1

print(f"val 변환 완료 - 성공: {success}, 건너뜀: {skip}")

# =====================
# 5. 이미지 symlink 생성
# =====================
for img in os.listdir(ts_path):
    dst = os.path.join('/kaggle/working/boar/images/train', img)
    if not os.path.exists(dst):
        os.symlink(os.path.join(ts_path, img), dst)

for img in os.listdir(vs_path):
    dst = os.path.join('/kaggle/working/boar/images/val', img)
    if not os.path.exists(dst):
        os.symlink(os.path.join(vs_path, img), dst)

print(f"train 이미지 symlink 수: {len(os.listdir('/kaggle/working/boar/images/train'))}")
print(f"val 이미지 symlink 수: {len(os.listdir('/kaggle/working/boar/images/val'))}")

# =====================
# 6. labels symlink 생성
# =====================
if not os.path.exists('/kaggle/working/boar/labels'):
    os.symlink('/kaggle/working/dataset/labels', '/kaggle/working/boar/labels')
print("labels symlink 생성 완료")

# =====================
# 7. dataset.yaml 생성
# =====================
yaml_content = """path: /kaggle/working/boar
train: images/train
val: images/val
nc: 1
names:
  0: wild_boar
"""
with open('/kaggle/working/boar/dataset.yaml', 'w') as f:
    f.write(yaml_content)
print("dataset.yaml 생성 완료")
print("===== 세팅 완료 =====")

train 변환 완료 - 성공: 28352, 건너뜀: 2348
val 변환 완료 - 성공: 3520, 건너뜀: 285
symlink 생성 완료
dataset.yaml 생성 완료
===== 세팅 완료 =====


# 2. 학습 시작!

In [4]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.1 MB/s eta 0:00:00a 0:00:01


In [15]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data='/kaggle/working/boar/dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    lr0=0.01,
    lrf=0.01,
    patience=20,
    augment=True,
    cos_lr=True,
    weight_decay=0.0005,
    device=0,
    project='/kaggle/working/runs',
    name='boar_detection',
    save=True,
    plots=True,
    cache=False
)

Ultralytics 8.4.53 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/boar/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=boar_detection-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pa

KeyboardInterrupt: 

> epoch 100개까지 다 돌리는건 너무 오래걸려서 20개정도 돌리고 학습 중단시킴
>
> 그 중 최고성능을 모델1로 최종 저장함
>
> 후에 데모에서 바로 불러와 사용할 수 있도록 로컬로 저장함!

In [16]:
import shutil
shutil.copy('/kaggle/working/runs/boar_detection-3/weights/best.pt',
            '/kaggle/working/boar_detection_best.pt')
print("저장 완료")

저장 완료
